In [9]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


FEATURES = ["P_laser", "v_scan", "d_laser", "t_powder"]
TARGETS = ["melt_width", "melt_depth", "melt_area"]
RISK = "balling"


def load_316l_csv(path: str | Path) -> pd.DataFrame:
    """
    Loads 316L_meltpool.csv.
    """
    path = Path(path)
    df = pd.read_csv(path, header=None, skiprows=1)
    if df.shape[1] != 9:
        raise ValueError(f"Expected 9 columns including ID, got {df.shape[1]}")

    df.columns = [
        "id", "P_laser", "v_scan", "d_laser", "t_powder",
        "melt_width", "melt_depth", "melt_area", "balling",
    ]

    numeric = df.columns
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="raise")
    df["id"] = df["id"].astype(int)
    df["balling"] = df["balling"].astype(np.float32)
    return df


@dataclass
class DataScalers:
    x: StandardScaler
    y: StandardScaler


def make_iid_split(
    df: pd.DataFrame,
    seed: int = 42,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Initial IID split only. Process-space split will be added separately."""
    train, temp = train_test_split(df, test_size=1.0 - train_ratio, random_state=seed)
    relative_val = val_ratio / (1.0 - train_ratio)
    val, test = train_test_split(temp, test_size=1.0 - relative_val, random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def fit_scalers(train_df: pd.DataFrame) -> DataScalers:
    x = StandardScaler().fit(train_df[FEATURES])
    y = StandardScaler().fit(train_df[TARGETS])
    return DataScalers(x=x, y=y)


class MeltPoolDataset(Dataset):
    def __init__(self, df: pd.DataFrame, scalers: DataScalers):
        self.df = df.reset_index(drop=True)
        self.x = scalers.x.transform(self.df[FEATURES]).astype(np.float32)
        self.y = scalers.y.transform(self.df[TARGETS]).astype(np.float32)
        self.balling = self.df[RISK].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "x": torch.from_numpy(self.x[idx]),
            "y": torch.from_numpy(self.y[idx]),
            "balling": torch.tensor(self.balling[idx]),
            "id": int(self.df.iloc[idx]["id"]),
        }


In [10]:
from __future__ import annotations

import torch
from torch import nn


class DeepSupervisionLoss(nn.Module):
    """
    L = L_data + alpha * L_physics
    """

    def __init__(self, gamma: float = 0.8):
        super().__init__()
        self.gamma = gamma

    def forward(self, predictions, target):
        total = 0.0
        weight_sum = 0.0

        for k, pred in enumerate(predictions):
            # Later iterations needed higher weight.
            weight = self.gamma ** (len(predictions) - 1 - k)
            total = total + weight * nn.functional.mse_loss(pred, target)
            weight_sum += weight

        return total / weight_sum


In [11]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass

import math
import torch


@dataclass
class PhysicsFeedback:
    residual: torch.Tensor
    confidence: torch.Tensor | None = None


class PhysicsEvaluator(ABC):
    @abstractmethod
    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        raise NotImplementedError


class ZeroPhysicsEvaluator(PhysicsEvaluator):
    """Architecture Prove-only"""

    def __init__(self, state_dim: int = 3):
        self.state_dim = state_dim

    def evaluate(self, process_x, predicted_state):
        return PhysicsFeedback(residual=torch.zeros_like(predicted_state))


class RosenthalPhysicsEvaluator(PhysicsEvaluator): #Rosenthal의 열역학 모델
    """
    Physics model:
        W = sqrt( 8/(pi*e) * eta*P / (rho*Cp*v*DeltaT) )
        D = W/2
        A = pi/4 * W * D
    """

    def __init__(
        self,
        x_mean,
        x_scale,
        y_mean,
        y_scale,
        absorptivity: float = 0.35,
        density: float = 7800.0,
        heat_capacity: float = 460.0,
        melting_temperature: float = 1678.0,
        preheat_temperature: float = 293.15,
        residual_clip: float = 5.0,
    ):
        self.x_mean = torch.as_tensor(x_mean, dtype=torch.float32)
        self.x_scale = torch.as_tensor(x_scale, dtype=torch.float32)
        self.y_mean = torch.as_tensor(y_mean, dtype=torch.float32)
        self.y_scale = torch.as_tensor(y_scale, dtype=torch.float32)

        self.absorptivity = absorptivity
        self.density = density
        self.heat_capacity = heat_capacity
        self.melting_temperature = melting_temperature
        self.preheat_temperature = preheat_temperature
        self.residual_clip = residual_clip

    def _physical_x(self, x: torch.Tensor) -> torch.Tensor:
        mean = self.x_mean.to(device=x.device, dtype=x.dtype)
        scale = self.x_scale.to(device=x.device, dtype=x.dtype)
        return x * scale + mean

    def _physical_y(self, y: torch.Tensor) -> torch.Tensor:
        mean = self.y_mean.to(device=y.device, dtype=y.dtype)
        scale = self.y_scale.to(device=y.device, dtype=y.dtype)
        return y * scale + mean

    def rosenthal_geometry(self, process_x: torch.Tensor) -> torch.Tensor:
        """Return [W_um, D_um, A_um2] from the analytical model."""
        x = self._physical_x(process_x)
        P_w = x[:, 0]
        v_mps = x[:, 1] / 1000.0  # dataset: mm/s -> m/s

        delta_T = self.melting_temperature - self.preheat_temperature
        width_m = torch.sqrt(
            (8.0 / (math.pi * math.e))
            * self.absorptivity
            * P_w
            / (self.density * self.heat_capacity * v_mps * delta_T)
        )
        depth_m = 0.5 * width_m
        area_m2 = math.pi / 4.0 * width_m * depth_m

        return torch.stack(
            [width_m * 1e6, depth_m * 1e6, area_m2 * 1e12], dim=-1
        )

    def evaluate(self, process_x: torch.Tensor, predicted_state: torch.Tensor) -> PhysicsFeedback:
        pred = self._physical_y(predicted_state)
        analytic = self.rosenthal_geometry(process_x)

        # 상대 residuals은 무차원이며 W, D, A 사이에서 Comparable.
        eps = torch.tensor(1e-6, device=pred.device, dtype=pred.dtype)
        residual = (pred - analytic) / (analytic.abs() + eps)
        residual = torch.clamp(residual, -self.residual_clip, self.residual_clip)

        # Confidence = 유한한 입력값에 대한 유효성 Mask.
        confidence = torch.ones_like(residual)
        confidence = torch.where(torch.isfinite(residual), confidence, torch.zeros_like(confidence))
        return PhysicsFeedback(residual=residual, confidence=confidence)


class DeltaFeedbackEvaluator(PhysicsEvaluator):
    """Debug 전용"""

    def __init__(self, state_dim: int = 3):
        self.state_dim = state_dim
        self.previous = None

    def reset(self):
        self.previous = None

    def evaluate(self, process_x, predicted_state):
        if self.previous is None:
            residual = torch.zeros_like(predicted_state)
        else:
            residual = predicted_state - self.previous
        self.previous = predicted_state.detach()
        return PhysicsFeedback(residual=residual)


In [12]:
# 2026. 601Solutions
# 유의해야할 점: 기존 Transformer 모델과 달리 이 Proj의 Transformer의 sequence는 데이터의 시간축이 아니라 추론 iteration이다.

"""
Mathmatical Modeling
x=[P,v,d,t]
y=[W,D,A]

x→fθ​(x)→y^​0​
r0​=PhysicsEvaluator(x,y^​0​)

h1​=Update(h0​,r0​)
^y​1​=fθ​(x,h1​)

r1​=PhysicsEvaluator(x,y^​1​)
Loop...
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch import nn


@dataclass
class LoopOutput:
    predictions: list[torch.Tensor]
    feedbacks: list[torch.Tensor]


class FeatureTokenizer(nn.Module):
    """
    스칼라값 to 토큰

    Input:
        [B, 4]

    Output:
        [B, 4, d_model]
    """

    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.scalar_projection = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(torch.randn(n_features, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.scalar_projection(x.unsqueeze(-1))
        return tokens + self.feature_embedding.unsqueeze(0)


class LoopedTransformer(nn.Module): # L/T Main

    def __init__(
        self,
        n_features: int = 4,
        state_dim: int = 3,
        feedback_dim: int = 3,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        loop_steps: int = 3,
    ):
        super().__init__()
        self.state_dim = state_dim
        self.feedback_dim = feedback_dim
        self.d_model = d_model
        self.loop_steps = loop_steps

        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.state_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.state_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, state_dim),
        )

        self.feedback_encoder = nn.Sequential(
            nn.Linear(feedback_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.state_update = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(
        self,
        x: torch.Tensor,
        physics_evaluator=None,
        loop_steps: int | None = None,
    ) -> LoopOutput:
        bsz = x.size(0)
        steps = loop_steps or self.loop_steps

        process_tokens = self.tokenizer(x)
        state = self.state_token.expand(bsz, -1, -1)

        predictions = []
        feedbacks = []

        for _ in range(steps):
            tokens = torch.cat([process_tokens, state], dim=1)
            encoded = self.transformer(tokens)

            state_hidden = encoded[:, -1, :]
            prediction = self.state_head(state_hidden)
            predictions.append(prediction)

            if physics_evaluator is None:
                feedback = torch.zeros_like(prediction)
            else:
                result = physics_evaluator.evaluate(x, prediction)
                feedback = result.residual

            feedbacks.append(feedback)

            feedback_hidden = self.feedback_encoder(feedback)
            state = state + self.state_update(
                torch.cat([state_hidden, feedback_hidden], dim=-1)
            ).unsqueeze(1)

        return LoopOutput(predictions=predictions, feedbacks=feedbacks)


class StateMLP(nn.Module):
    """baseline."""

    def __init__(self, n_features=4, state_dim=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, state_dim),
        )

    def forward(self, x):
        return self.net(x)




In [15]:
from __future__ import annotations

import argparse
from pathlib import Path

import torch
from torch.utils.data import DataLoader


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default=Path("/content/316L_meltpool.csv"))
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--loop-steps", type=int, default=3)
    parser.add_argument("--seed", type=int, default=42)
    args, unknown = parser.parse_known_args()

    torch.manual_seed(args.seed)

    df = load_316l_csv(args.data)
    train_df, val_df, test_df = make_iid_split(df, seed=args.seed)
    scalers = fit_scalers(train_df)

    train_loader = DataLoader(
        MeltPoolDataset(train_df, scalers),
        batch_size=args.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        MeltPoolDataset(val_df, scalers),
        batch_size=args.batch_size,
    )

    model = LoopedTransformer(loop_steps=args.loop_steps)
    physics = RosenthalPhysicsEvaluator(
        x_mean=scalers.x.mean_, x_scale=scalers.x.scale_,
        y_mean=scalers.y.mean_, y_scale=scalers.y.scale_,
    )
    criterion = DeepSupervisionLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)

    best_val = float("inf")

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch["x"], physics_evaluator=physics)
            loss = criterion(out.predictions, batch["y"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * batch["x"].size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch["x"], physics_evaluator=physics)
                loss = criterion(out.predictions, batch["y"])
                val_loss += loss.item() * batch["x"].size(0)

        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {
                    "model": model.state_dict(),
                    "x_scaler": scalers.x,
                    "y_scaler": scalers.y,
                },
                "looped_transformer_best.pt",
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} "
                f"train={train_loss:.5f} "
                f"val={val_loss:.5f}"
            )

    print("Training complete.")
    print(f"Best validation loss: {best_val:.5f}")
    print(
        "NOTE: RosenthalPhysicsEvaluator is an analytical consistency model. "
        "It is a low-order conduction-mode approximation, not ground truth."
    )


if __name__ == "__main__":
    main()


/tmp/ipykernel_1800/2566090003.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


epoch=001 train=0.97730 val=0.64973
epoch=010 train=0.39769 val=0.34374
epoch=020 train=0.22422 val=0.24205
epoch=030 train=0.16470 val=0.15438
epoch=040 train=0.13272 val=0.11331
epoch=050 train=0.12826 val=0.11196
epoch=060 train=0.12270 val=0.10997
epoch=070 train=0.10627 val=0.09270
epoch=080 train=0.10243 val=0.09860
epoch=090 train=0.10490 val=0.09864
epoch=100 train=0.10243 val=0.10009
Training complete.
Best validation loss: 0.09270
NOTE: RosenthalPhysicsEvaluator is an analytical consistency model. It is a low-order conduction-mode approximation, not ground truth.


In [14]:
# @title
from __future__ import annotations

import argparse
from pathlib import Path

import torch
from torch.utils.data import DataLoader


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default=Path("/content/316L_meltpool.csv"))
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--loop-steps", type=int, default=3)
    parser.add_argument("--seed", type=int, default=42)
    args, unknown = parser.parse_known_args()

    torch.manual_seed(args.seed)

    df = load_316l_csv(args.data)
    train_df, val_df, test_df = make_iid_split(df, seed=args.seed)
    scalers = fit_scalers(train_df)

    train_loader = DataLoader(
        MeltPoolDataset(train_df, scalers),
        batch_size=args.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        MeltPoolDataset(val_df, scalers),
        batch_size=args.batch_size,
    )

    model = LoopedTransformer(loop_steps=args.loop_steps)
    physics = ZeroPhysicsEvaluator(state_dim=3)
    criterion = DeepSupervisionLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)

    best_val = float("inf")

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch["x"], physics_evaluator=physics)
            loss = criterion(out.predictions, batch["y"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * batch["x"].size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch["x"], physics_evaluator=physics)
                loss = criterion(out.predictions, batch["y"])
                val_loss += loss.item() * batch["x"].size(0)

        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {
                    "model": model.state_dict(),
                    "x_scaler": scalers.x,
                    "y_scaler": scalers.y,
                },
                "looped_transformer_best.pt",
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} "
                f"train={train_loss:.5f} "
                f"val={val_loss:.5f}"
            )

    print("Training complete.")
    print(f"Best validation loss: {best_val:.5f}")
    print(
        "NOTE: ZeroPhysicsEvaluator is architecture-only. "
        "Do not interpret this run as physics-aware training."
    )


if __name__ == "__main__":
    main()


/tmp/ipykernel_1800/2566090003.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


epoch=001 train=1.14168 val=0.87077
epoch=010 train=0.80733 val=0.73716
epoch=020 train=0.30502 val=0.29561
epoch=030 train=0.22931 val=0.18134
Training complete.
Best validation loss: 0.18134
NOTE: ZeroPhysicsEvaluator is architecture-only. Do not interpret this run as physics-aware training.


In [17]:
import torch
import pandas as pd
import numpy as np

# 1. 저장된 체크포인트 로드 (weights_only=False 추가하여 StandardScaler 로드 허용)
checkpoint_path = "looped_transformer_best.pt"
checkpoint = torch.load(checkpoint_path, weights_only=False)

# 2. 데이터셋 로드 및 분할
df = load_316l_csv("/content/316L_meltpool.csv")
train_df, val_df, test_df = make_iid_split(df, seed=42)

# 저장되어 있던 Scaler 및 가중치 복원
scaler_x = checkpoint["x_scaler"]
scaler_y = checkpoint["y_scaler"]

# 3. 모델 초기화 및 가중치 적용
model = LoopedTransformer(loop_steps=3)
model.load_state_dict(checkpoint["model"])
model.eval()

# 4. 검증 데이터 준비
# 수동으로 스케일링된 x 텐서 생성
x_scaled = scaler_x.transform(val_df[FEATURES]).astype(np.float32)
x_tensor = torch.from_numpy(x_scaled)

# 5. 예측 수행
with torch.no_grad():
    output = model(x_tensor)
    # LoopedTransformer의 최종 루프 스텝(index: -1) 예측값 선택
    final_predictions_scaled = output.predictions[-1].numpy()

# 6. 역변환 (Inverse Transform)을 통한 원래 스케일 복원
final_predictions = scaler_y.inverse_transform(final_predictions_scaled)

# 7. 실제값과 예측값 비교용 DataFrame 생성
comparison_df = pd.DataFrame()
for i, target_col in enumerate(TARGETS):
    comparison_df[f"Actual_{target_col}"] = val_df[target_col]
    comparison_df[f"Pred_{target_col}"] = final_predictions[:, i]
    comparison_df[f"Error_{target_col}"] = comparison_df[f"Actual_{target_col}"] - comparison_df[f"Pred_{target_col}"]

# 상위 10개 결과 출력
print("--- Validation Set Predictions vs Ground Truth (First 10 Rows) ---")
display(comparison_df.head(10).round(4))


--- Validation Set Predictions vs Ground Truth (First 10 Rows) ---


/tmp/ipykernel_1800/2566090003.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


,Actual_melt_width,Pred_melt_width,Error_melt_width,Actual_melt_depth,Pred_melt_depth,Error_melt_depth,Actual_melt_area,Pred_melt_area,Error_melt_area
0,117.300,130.228806,-12.9289,96.700,57.165298,39.5347,9203.400,9334.480469,-131.0805
1,232.600,290.748291,-58.1483,264.300,434.197998,-169.8980,56199.400,87208.507812,-31009.1078
2,85.900,102.086800,-16.1868,57.200,43.237999,13.9620,4268.700,4790.069336,-521.3693
3,111.900,135.505707,-23.6057,226.400,339.338196,-112.9382,20580.300,36330.613281,-15750.3133
4,109.651,114.684799,-5.0338,108.177,75.404602,32.7724,8134.272,8614.805664,-480.5337
5,156.742,145.299301,11.4427,33.563,39.993301,-6.4303,7913.787,8307.135742,-393.3487
6,105.402,113.977798,-8.5758,82.524,60.066502,22.4575,8047.975,7725.824219,322.1508
7,209.300,235.620605,-26.3207,178.200,185.740402,-7.5404,36661.000,39711.824219,-3050.8242
8,222.000,239.265503,-17.2655,184.700,390.773590,-206.0737,38844.700,65290.484375,-26445.7844
9,136.700,166.299606,-29.5996,327.500,441.956604,-114.4566,35899.000,54464.613281,-18565.6172
